In [0]:
# configure paths
catalog = "global_mart_retail_dev"
bronze_table = "superstore_orders"
silver_table = "geography"

# table path
bronze_path = f"{catalog}.bronze.{bronze_table}"
silver_path = f"{catalog}.silver.{silver_table}"

print(f"Bronze table path: {bronze_path}")
print(f"Silver table path: {silver_path}")

In [0]:
# import libraries
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# read bronze data 
bronze_df = spark.read.table(bronze_path)

# creating silver table for  customers 
df_clean_geography = (                   
    bronze_df
    .select(trim(col("City")).alias("city"),
            trim(col("State")).alias("state"),
            trim(col("Postal_Code")).alias("postal_code"),
            trim(col("Country")).alias("country"),
            trim(col("Region")).alias("region")).dropDuplicates(["country","region","state","city","postal_code"])
    )
    
print(f"total count: {df_clean_geography.count()}") # count after dedeuplication

display(df_clean_geography.limit(5))

In [0]:
# generate hash on cleaned values to track changes

df_geography_stage = (
    df_clean_geography
    .withColumn("geography_hash", sha2(concat_ws("|",col("city"),col("state"),col("postal_code"),col("country"),col("region")),256))
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to", lit("9999-12-31").cast("timestamp"))
    .withColumn("is_current_flag",lit(True))
    .withColumn("load_timestamp", current_timestamp())

)

display(df_geography_stage)

In [0]:
# Create the silver geography table if it does not exist
spark.sql(
    """
        CREATE TABLE IF NOT EXISTS global_mart_retail_dev.silver.geography
        (
            postal_code string not null,
            city string,
            state string,
            country string,
            region string,
            geography_hash string not null ,
            valid_from timestamp not null,
            valid_to timestamp,
            is_current_flag boolean not null,
            load_timestamp timestamp not null
        )
        USING DELTA
    """
)


# Set up SCD Type 2 merge for customer table
geography_table = DeltaTable.forName(spark,"global_mart_retail_dev.silver.geography")

(
    geography_table.alias("target")
    .merge(
        df_geography_stage.alias("source"),
        "target.postal_code = source.postal_code AND target.is_current_flag = true"
    )
    # Expire current records when any attributes have changed
    .whenMatchedUpdate(
        condition="target.geography_hash <> source.geography_hash",
        set={
            "valid_to": "current_timestamp()",
            "is_current_flag": "false"
        }
    )
    # Insert new records or updated records
    .whenNotMatchedInsert(
        values={
            "postal_code": "source.postal_code",
            "city": "source.city",
            "state": "source.state",
            "country": "source.country",
            "region": "source.region",
            "geography_hash": "source.geography_hash",
            "valid_from": "source.valid_from",
            "valid_to": "source.valid_to",
            "is_current_flag": "source.is_current_flag",
            "load_timestamp": "source.load_timestamp"
        }
    )
    .execute()
)

In [0]:
%sql 

SELECT * FROM  global_mart_retail_dev.silver.geography 


In [0]:
%sql
SELECT * FROM  global_mart_retail_dev.silver.geography 
where postal_code = '60540'